In [1]:
from pathlib import Path
import re
import gc

import torch
import pandas as pd

In [2]:
def _parse_dir_name(dir_name: str):
    match = re.match(r"^attribution__(.+?)__(.+)$", dir_name)
    if not match:
        return None, None
    return match.group(1), match.group(2)


def _parse_method_name(file_stem: str):
    if "_" not in file_stem:
        return None, file_stem
    attribution_type, method_name = file_stem.split("_", 1)
    return attribution_type, method_name


def collect_attribution_metadata(results_root: str):
    results_root = Path(results_root)
    rows = []

    for attr_dir in sorted(results_root.glob("attribution__*__*")):
        if not attr_dir.is_dir():
            continue

        model_name, dataset_name = _parse_dir_name(attr_dir.name)

        for pt_path in sorted(attr_dir.glob("*.pt")):
            attribution_type, method_name = _parse_method_name(pt_path.stem)

            try:
                payload = torch.load(pt_path, map_location="cpu")
            except Exception as exc:
                rows.append({
                    "model": model_name,
                    "dataset": dataset_name,
                    "attribution_type": attribution_type,
                    "method": method_name,
                    "file_name": pt_path.name,
                    "file_path": str(pt_path),
                    "load_error": str(exc),
                })
                continue

            metadata = payload.get("metadata", {}) if isinstance(payload, dict) else {}

            rows.append({
                "model": model_name,
                "dataset": dataset_name,
                "attribution_type": attribution_type,
                "method": method_name,
                "file_name": pt_path.name,
                "file_path": str(pt_path),
                "batch_size": metadata.get("batch_size"),
                "time_sec": metadata.get("time_sec"),
                "peak_memory_allocated_mb": metadata.get("peak_memory_allocated_mb"),
                "peak_memory_reserved_mb": metadata.get("peak_memory_reserved_mb"),
                "load_error": None,
            })

            del payload
            gc.collect()

    df = pd.DataFrame(rows)

    if not df.empty:
        sort_cols = [
            col for col in ["dataset", "model", "attribution_type", "method"] if col in df.columns
        ]
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df


def export_runtime_memory_csv(results_root: str, output_csv: str):
    df = collect_attribution_metadata(results_root)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)

    return df, output_csv

In [3]:
results_root = "/mnt/dacslab/dpa/results"
output_csv = "/home/dacslab/djk/DPA/DPA_Tracing/empirical_results/attribution_runtime_memory.csv"

df, saved_path = export_runtime_memory_csv(results_root, output_csv)

print(f"Saved CSV: {saved_path}")
print(f"Rows: {len(df)}")

df.head(20)

Saved CSV: /home/dacslab/djk/DPA/DPA_Tracing/empirical_results/attribution_runtime_memory.csv
Rows: 275


,model,dataset,attribution_type,method,file_name,file_path,batch_size,time_sec,peak_memory_allocated_mb,peak_memory_reserved_mb,load_error
0,meta_llama_llama_2_7b_chat_hf,imdb,input,ap,input_ap.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,1255.7440,13330.01,13950.0,None
1,meta_llama_llama_2_7b_chat_hf,imdb,input,attn_rollout,input_attn_rollout.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,16.9640,13632.82,15034.0,None
2,meta_llama_llama_2_7b_chat_hf,imdb,input,depass,input_depass.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,1134.4690,14130.55,14710.0,None
3,meta_llama_llama_2_7b_chat_hf,imdb,input,dpa_025_025_05_05_05,input_dpa_025_025_05_05_05.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,36.3344,15932.82,16998.0,None
4,meta_llama_llama_2_7b_chat_hf,imdb,input,gradient,input_gradient.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,33.1647,18272.30,19844.0,None
5,meta_llama_llama_2_7b_chat_hf,imdb,input,ifr,input_ifr.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,1293.8082,13503.09,14648.0,None
6,meta_llama_llama_2_7b_chat_hf,imdb,input,input_x_gradient,input_input_x_gradient.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,33.2761,19162.76,21198.0,None
7,meta_llama_llama_2_7b_chat_hf,imdb,input,integrated_gradient,input_integrated_gradient.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,616.5266,19269.34,20638.0,None
8,meta_llama_llama_2_7b_chat_hf,imdb,input,last_layer_attn,input_last_layer_attn.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,16.6896,13632.82,15034.0,None
9,meta_llama_llama_2_7b_chat_hf,imdb,input,mean_attn,input_mean_attn.pt,/mnt/dacslab/dpa/results/attribution__meta_lla...,4,16.8413,13632.82,15034.0,None


In [4]:
summary_cols = ["time_sec", "peak_memory_allocated_mb", "peak_memory_reserved_mb"]
valid_df = df[df["load_error"].isna()].copy() if "load_error" in df.columns else df.copy()

if not valid_df.empty:
    summary_df = (
        valid_df
        .groupby(["dataset", "attribution_type", "method"], dropna=False)[summary_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(["dataset", "attribution_type", "method"])
    )
    display(summary_df.head(30))
else:
    print("No valid metadata rows found.")

,dataset,attribution_type,method,time_sec,peak_memory_allocated_mb,peak_memory_reserved_mb
0,imdb,input,ap,1276.107900,12731.5925,13347.5
1,imdb,input,attn_rollout,18.581725,13003.6325,14193.5
2,imdb,input,depass,1144.844575,13537.8600,14778.5
3,imdb,input,dpa_025_025_05_05_05,42.552150,15412.9150,16841.0
4,imdb,input,gradient,36.781525,17974.8875,19306.5
5,imdb,input,ifr,1768.627800,12824.0675,13761.5
6,imdb,input,input_x_gradient,36.947400,18662.8825,20362.0
7,imdb,input,integrated_gradient,690.000400,18755.4900,19899.5
8,imdb,input,last_layer_attn,18.389800,13001.7100,14193.5
9,imdb,input,mean_attn,18.451700,13087.6450,14253.0
